# Free-sample pilot: chi-square test on a 2 x 2 table

**Test of Independence** · a worked example from the StatNex teaching guide

**The story.** A snack brand ran a small in-store pilot: 11 shoppers were offered a free sample and 11 were not, and staff noted who bought a pack. The sample is small, so the result needs care.

**The question.** Is buying related to getting a free sample?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `04_crosstab/independence_test__2x2.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Group,Bought,Did not buy,Customers
Free sample,8.0,3.0,11.0
No sample,2.0,9.0,11.0
Total,10.0,12.0,22.0

These cells save `totals.py`: a step that no library does, written out. The analysis imports it.

In [ ]:
%%writefile totals.py
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import optimize, stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# A table of counts as printed: one column per category of the column variable, one
# row per category of the row variable. Rows with a blank count are skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Bought", "Did not buy"]
table = data[columns].apply(pd.to_numeric, errors="coerce")
table.index = data["Group"].astype(str)
# A total row or column (Total, or its Japanese equivalent) is a margin, not a category: counting it would double
# every count, so it is left out.
# Spaces, underscores, case and full-width letters are ignored when names are compared
# (totals.py).
from totals import is_total

table = table[[name for name in columns if not is_total(name)]]
table = table[[not is_total(name) for name in table.index]]
table = table[table.notna().all(axis=1)]
observed = table.to_numpy(float)

## Observed frequencies

In [ ]:
grid = np.vstack([observed, observed.sum(axis=0)])
grid = np.column_stack([grid, grid.sum(axis=1)])
observed_table = pd.DataFrame(grid, index=[*table.index, "Total"], columns=[*table.columns, "Total"])
print(observed_table)

## Expected frequencies and the chi-square test

In [ ]:
# Under independence a cell expects (row total x column total) / n.
chi2, p, df, expected = stats.chi2_contingency(observed, correction=False)
expected_table = pd.DataFrame(expected, index=table.index, columns=table.columns)
print()
print(expected_table)
if (expected < 5).any():
    print(f"\n{(expected < 5).sum()} cells expect fewer than 5: consider Fisher's exact test.")

two_by_two = observed.shape == (2, 2)
if two_by_two:
    # On a 2x2 table, Yates' continuity correction is reported as well.
    chi2_yates, p_yates, _, _ = stats.chi2_contingency(observed, correction=True)
    test = pd.DataFrame([[chi2, df, p], [chi2_yates, df, p_yates]],
                        index=["Uncorrected", "Yates' correction"], columns=["chi-square", "df", "P"])
else:
    test = pd.DataFrame([[chi2, df, p]], index=["Pearson"], columns=["chi-square", "df", "P"])
print()
print(test)

## Strength of association

In [ ]:
# Cramer's V rescales the chi-square to 0..1 whatever the table's shape. On a 2x2 table
# Yule's Q is the odds ratio mapped onto -1..1.
# How to read it: the chi-square's P says whether there is a link at all; Cramer's V
# says how strong it is (0 = none, 1 = perfect) and, unlike the chi-square, does not
# grow just because the sample is bigger. The chi-square needs expected counts of
# about 5 or more in every cell; below that, use Fisher's exact test.
n = observed.sum()
effect = {"Cramer's V": np.sqrt(chi2 / (n * (min(observed.shape) - 1)))}
if two_by_two:
    (a, b), (c, d) = observed
    effect["Yule's Q"] = (a * d - b * c) / (a * d + b * c) if a * d + b * c else np.nan
effect = pd.Series(effect)
print()
print(effect)

## Effect sizes

In [ ]:
# A 95% interval for Cramer's V. Under an association the chi-square follows a noncentral
# chi-square on the same df. The interval for its noncentrality lambda is the pair of lambda
# that put the observed chi-square at the upper and lower 2.5% points of their distributions
# (Steiger & Fouladi 1997), and each converts to V as the chi-square itself does,
# sqrt(lambda / (n (min(r, c) - 1))), capped at 1. V is read on Cohen's (1988) scale for
# w = V sqrt(min(r, c) - 1): 0.1 small, 0.3 medium, 0.5 large.


def noncentrality(p):
    """The lambda at which P(noncentral chi-square <= the observed one) = p; 0 if none."""
    if stats.chi2.cdf(chi2, df) <= p:
        return 0.0
    high = max(2 * chi2, 1.0)
    while stats.ncx2.cdf(chi2, df, high) > p:
        high *= 2
    return optimize.brentq(lambda lam: stats.ncx2.cdf(chi2, df, lam) - p, 0.0, high,
                           xtol=1e-13, rtol=4 * np.finfo(float).eps, maxiter=500)


scale = n * (min(observed.shape) - 1)
rows = {"Cramer's V": [effect["Cramer's V"], np.sqrt(noncentrality(0.975) / scale),
                       min(1.0, np.sqrt(noncentrality(0.025) / scale))]}
if two_by_two:
    # The odds ratio ad/bc with Woolf's (1955) interval on the log scale. An empty cell
    # leaves both undefined, so then 0.5 is added to every cell first (Haldane 1956). Read on
    # Chen, Cohen & Chen (2010): 1.68 small, 3.47 medium, 6.71 large (1/OR below 1).
    cells = observed + 0.5 if (observed == 0).any() else observed
    (a, b), (c, d) = cells
    odds = a * d / (b * c)
    half = stats.norm.ppf(0.975) * np.sqrt(1 / a + 1 / b + 1 / c + 1 / d)
    rows["odds ratio (Woolf)"] = [odds, odds * np.exp(-half), odds * np.exp(half)]
effect_sizes = pd.DataFrame.from_dict(rows, orient="index",
                                      columns=["value", "95% lower", "95% upper"])
print()
print(effect_sizes)

## What the result says

In the pilot, 8 of 11 shoppers with a free sample bought, against 2 of 11 without. With no effect of the sample, we would expect 5 buyers in each group.

The chi-square test measures this gap. Uncorrected, chi-square is 6.6 with P = 0.010. With Yates' correction, a more careful version for small tables, it is 4.58 with P = 0.032. A P value is the chance of a gap at least this large if the sample made no difference. Both are below 0.05, so buying is related to the free sample.

The size of the effect is uncertain. The odds ratio is 12, but its confidence interval, the range of plausible values, runs from 1.58 to 91. With 22 shoppers, the pilot shows a link, not its size or its cause.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.